In [16]:
FILE_ID =  5422

In [22]:
files_dict = {
    406: {
        "main_padding": "./php/406/406_whole.php",
        "buggy_content": "./php/406/406_smallestBuggy.php",
        "buggy_line": "$selectedIds = explode(',', $selectedIds);",
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    408: {
        'main_padding': './php/408/408_whole.php',
        "additional_padding": ["./php/408/additional_padding.php"],
        "buggy_content": "./php/408/408_smallestBuggy.php",
        "buggy_line": "'SELECT * FROM ' . static::table_name() . ' WHERE ' . $property .  ' = \'' . $value . '\' LIMIT 0,1'",
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    405: {
        'main_padding': './php/405/405_whole.php',
        "additional_padding": ["./php/405/additional_padding.php"],
        "buggy_content": "./php/405/405_smallestBuggy.php",
        "buggy_line": '''$where = "WHERE group_ID = {$group_id}";''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    4921: {
        'main_padding': './go/4921/4921_modifiedFile.go',
        "additional_padding": ["./go/4921/additional_padding.go"],
        "buggy_content": "./go/4921/4921_smallestBuggy.go",
        "buggy_line": '''order := fmt.Sprintf("`%s` %s", DefaultQuery(c, "sort_by", "id"), sort)''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    4928: {
        'main_padding': './php/4928/4928_whole.php',
        "additional_padding": ["./php/4928/additional_padding.php"],
        "buggy_content": "./php/4928/4928_smallestBuggy.php",
        "buggy_line": '''return @mysqli_real_escape_string($fmdb->dbh, $data);''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    3989:{
        'main_padding': './js/3989/3989_whole.js',
        "additional_padding": ["./js/3989/additional_padding.js"],
        "buggy_content": "./js/3989/3989_smallestBuggy.js",
        "buggy_line": '''`window[${idJSON}].push(${serializedCacheArgs});`,''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-79"
    },
    3976:{
        'main_padding': './js/3976/3976_whole.js',
        "additional_padding": ["./js/3976/additional_padding.js"],
        "buggy_content": "./js/3976/3976_smallestBuggy.js",
        "buggy_line": '''const header = container.querySelector(`h${level}`);''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-79"
    },
    3968:{
        'main_padding': './php/3968/3968_whole.php',
        "additional_padding": ["./php/3968/additional_padding.php"],
        "buggy_content": "./php/3968/3968_smallestBuggy.php",
        "buggy_line": '''$response['data']['path'] = $model->path;''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-79"
    },
    300:{
        'main_padding': './php/300/300_modifiedFile.php',
        "additional_padding": ["./php/300/additional_padding.php"],
        "buggy_content": "./php/300/300_smallestBuggy.php",
        "buggy_line": '''<td>' . $item->id . '</td>''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-79"
    },
    302:{
        'main_padding': './rb/302/302_modifiedFile.rb',
        "additional_padding": ["./rb/302/additional_padding.rb"],
        "buggy_content": "./rb/302/302_smallestBuggy.rb",
        "buggy_line": '''@filter = params[:filter] || "*"''',
        "split_string": "# -x-",
        "CWE_ID": "CWE-79"
    },
    5422:{
        'main_padding': './go/5422/5422_modifiedFile.go',
        "additional_padding": ["./go/5422/additional_padding.go"],
        "buggy_content": "./go/5422/5422_smallestBuggy.go",
        "buggy_line": '''outdir := filepath.Join(basePath, name)''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-22"
    }
}

In [3]:
def relaxed_knapsack(items, capacity, tolerance=0):
    # Dynamic programming table
    dp = [[0] * (capacity + tolerance + 1) for _ in range(len(items) + 1)]
    for i in range(1, len(items) + 1):
        item_size = items[i - 1]['size']
        for j in range(capacity + tolerance + 1):
            if item_size <= j:
                dp[i][j] = max(dp[i - 1][j], dp[i - 1][j - item_size] + item_size)
            else:
                dp[i][j] = dp[i - 1][j]
    # Find the best value close to capacity
    best_value = max(dp[len(items)][capacity:capacity + tolerance + 1])
    # Backtrack to find selected items
    result = []
    w = dp[len(items)].index(best_value)
    for i in range(len(items), 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            result.append(items[i - 1])
            w -= items[i - 1]['size']
    return result

def parse_file(filename, split_string):
    snippets = []
    inside_snippet = False
    snippet_content = []

    with open(filename, 'r') as file:
        for line in file:
            # print(line)
            if split_string in line:
                # print("this line has split string")
                if inside_snippet:
                    # End of snippet
                    snippet = ''.join(snippet_content).strip()
                    snippets.append({
                        'size': len(snippet),
                        'snippet': snippet
                    })
                    snippet_content = []
                    inside_snippet = True
                else:
                    inside_snippet = not inside_snippet
            elif inside_snippet:
                snippet_content.append(line)
    
    return snippets

def get_padding_content(file_id, filename):
    file = files_dict[file_id]
    return parse_file(filename, file['split_string'])

In [4]:
def make_file_with_padding(file_id, target_chars, target_bug_position):
    # assert that bug position is not larger than target chars
    assert target_bug_position < target_chars
    # lets assume that we allow only mod 500 for both inputs
    assert target_bug_position % 500 == 0
    assert target_chars % 500 == 0

    with open(files_dict[file_id]['buggy_content'], 'r') as file:
        content = file.read()

    snippets = get_padding_content(file_id, files_dict[file_id]['main_padding'])
    buggy_chars = len(content) - len("{prepend_content}{append_content}")
    
    # find out how many 
    prepend_chars = target_bug_position
    append_chars = target_chars - prepend_chars - buggy_chars

    # count size of all snippets
    total_count = sum(x['size'] for x in snippets)

    print(len(snippets))
    if(total_count < prepend_chars + append_chars):
        for file in files_dict[file_id]['additional_padding']:
            snippets.extend(get_padding_content(file_id,file ))
    print(len(snippets))

    # TODO add how many padding comes from other files and from old file (can have also some inpact on something)

    # generate snippets
    prepend_snippets = relaxed_knapsack(snippets, prepend_chars, 50) if prepend_chars > 50 else {}
    append_snippets = relaxed_knapsack(snippets, append_chars, 50) if append_chars > 50 else {}

    # generate the final content
    prepend_content = '\n'.join([snippet['snippet'] for snippet in prepend_snippets])
    append_content = '\n'.join([snippet['snippet'] for snippet in append_snippets])

    # insert into content (replace prepend_content and append_content)
    content = content.replace('{prepend_content}', prepend_content, 1)
    content = content.replace('{append_content}', append_content, 1)
    print(f"Prepend chars: {prepend_chars}")
    print(f"Append chars: {append_chars}")
    print(f"buggy length: {buggy_chars}")
    print(f"Prepend snippets: {len(prepend_content)}")
    print(f"Append snippets: {len(append_content)}")


    return content


In [25]:
get_padding_content(FILE_ID, files_dict[FILE_ID]['additional_padding'][0])

[{'size': 5049,
  'snippet': '/*\nCopyright The Helm Authors.\nLicensed under the Apache License, Version 2.0 (the "License");\nyou may not use this file except in compliance with the License.\nYou may obtain a copy of the License at\n\nhttp://www.apache.org/licenses/LICENSE-2.0\n\nUnless required by applicable law or agreed to in writing, software\ndistributed under the License is distributed on an "AS IS" BASIS,\nWITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\nSee the License for the specific language governing permissions and\nlimitations under the License.\n*/\n\npackage chart\n\nimport (\n\t"path/filepath"\n\t"regexp"\n\t"strings"\n)\n\n// APIVersionV1 is the API version number for version 1.\nconst APIVersionV1 = "v1"\n\n// APIVersionV2 is the API version number for version 2.\nconst APIVersionV2 = "v2"\n\n// aliasNameFormat defines the characters that are legal in an alias name.\nvar aliasNameFormat = regexp.MustCompile("^[a-zA-Z0-9_-]+$")\n\n// Chart i

In [26]:
file = make_file_with_padding(FILE_ID, 30000, 500)
print(len(file))
print(file)

1
9
Prepend chars: 500
Append chars: 29391
buggy length: 109
Prepend snippets: 0
Append snippets: 29416
29525

func getOutDir(basePath, name string) string {
	outdir := filepath.Join(basePath, name)
	return outdir
}
/*
Copyright The Helm Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
*/

package loader

import (
	"bytes"
	"log"
	"os"
	"path/filepath"
	"strings"

	"github.com/pkg/errors"
	"sigs.k8s.io/yaml"

	"helm.sh/helm/v3/pkg/chart"
)

// ChartLoader loads a chart.
type ChartLoader interface {
	Load() (*c

In [7]:
range1 = range(500, 28000, 500)

for i in range1:
    file = make_file_with_padding(FILE_ID,28000, i )
    if(abs(len(file)-28000) > 300):
        print("*"*20)
    print(f"expected length: {28000} and got {len(file)}")

15
45
Prepend chars: 500
Append chars: 26933
buggy length: 567
Prepend snippets: 553
Append snippets: 27017
expected length: 28000 and got 28137
15
45
Prepend chars: 1000
Append chars: 26433
buggy length: 567
Prepend snippets: 1053
Append snippets: 26516
expected length: 28000 and got 28136
15
45
Prepend chars: 1500
Append chars: 25933
buggy length: 567
Prepend snippets: 1552
Append snippets: 26012
expected length: 28000 and got 28131
15
45
Prepend chars: 2000
Append chars: 25433
buggy length: 567
Prepend snippets: 2053
Append snippets: 25514
expected length: 28000 and got 28134
15
45
Prepend chars: 2500
Append chars: 24933
buggy length: 567
Prepend snippets: 2555
Append snippets: 25011
expected length: 28000 and got 28133
15
45
Prepend chars: 3000
Append chars: 24433
buggy length: 567
Prepend snippets: 3052
Append snippets: 24515
expected length: 28000 and got 28134
15
45
Prepend chars: 3500
Append chars: 23933
buggy length: 567
Prepend snippets: 3553
Append snippets: 24012
expected l

In [8]:
# create files with 30'000 characters with different bug positions, each should have ist own ID where FILEID_CHARS_BUGPOSITION